# Logistic Regression Imbalanced

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from collections import Counter

In [ ]:
# create
X,y=make_classification(
    n_samples=10000,
    n_features=2,
    n_clusters_per_class=2,
    n_redundant=0,
    weights=[0.99],
    random_state=67,
)

In [ ]:
X=pd.DataFrame(X)
X

In [ ]:
Counter(y)

In [ ]:
columns = X.columns
columnsL = list(X.columns)
columnsL

In [ ]:
sns.scatterplot(data=X, x=X[0],y=X[1], hue=y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train, y_test = train_test_split(
    X,y,test_size = 0.25,
    random_state = 0
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
# now we define the parameters we want to mess around with
# refer to the model's documentation to see the options and
# select the ones we want to search around in, for eg:
penalty = ['l1','l2','elasticnet']
c_values = [100,10,1.0,0.1,0.01]
solver = ['newton-cg', 'lbfgs', 'liblinear','sag','saga']
max_iter = [200,400,600,800,1000]
class_weight = [{0:w,1:y} for w in [x for x in range(0,110,10)] for y in [x for x in range(0,110,10)]]
class_weighted = [{0:w,1:y} for w in [1,10,50,100] for y in [1,10,50,100]]
params = dict(penalty = penalty, C = c_values, solver=solver, max_iter = max_iter)



In [ ]:
from itertools import product
class_labels = np.unique(y)  # Automatically fetch unique class labels
weights_range = range(0, 110, 10)
# Now use this dynamic class_labels in the previous code
class_weights = [{cls: w for cls, w in zip(class_labels, weights_combination)}
                 for weights_combination in product(weights_range, repeat=len(class_labels))]


In [ ]:
param_grid = [
    # L1 penalty: only liblinear and saga
    {
        'penalty': ['l1'],
        'solver': ['liblinear', 'saga'],
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': max_iter,
        'class_weight':class_weighted
    },
    # L2 penalty: works with most solvers
    {
        'penalty': ['l2'],
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': max_iter,
        'class_weight':class_weighted
    },
    # ElasticNet: only saga, needs l1_ratio
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'l1_ratio': [0.3, 0.5, 0.7],  # mix of L1/L2
        'C': [0.01, 0.1, 1, 10, 100],
        'max_iter': max_iter,
        'class_weight':class_weighted
    }
]

from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold()
#GridSearchCV
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(estimator= model,
                    param_grid= param_grid,
                    scoring='f1',
                    cv=cv,
                    n_jobs=-1
                    )


In [ ]:
class_weighted

In [ ]:
print(grid)

In [ ]:
grid.fit(X_train,y_train)
grid.best_params_

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Randomly sample from the parameter grid
random_search = RandomizedSearchCV(estimator=LogisticRegression(),
                                   param_distributions=param_grid,
                                   scoring='f1',
                                   cv=cv,
                                   n_jobs=-1,
                                   n_iter=50,  # Number of random combinations to test
                                   random_state=42)

# Fit the model
random_search.fit(X_train, y_train)

# Get the best parameters
print(random_search.best_params_)


In [ ]:
y_pred = grid.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, auc

In [ ]:
from sklearn.metrics import confusion_matrix


score = accuracy_score(y_pred, y_test)
print(score)
print(classification_report(y_pred, y_test))
print(confusion_matrix(y_pred,y_test))

## Understanding ROC and AUC

In [ ]:
X,y = make_classification(
    n_samples = 1000,
    n_classes = 2,
    random_state = 22
)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25, random_state=22)

In [ ]:
# getting a dummy model with default 0 output
dummy_model_prob = [0 for _ in range(len(y_test))]
dummy_model_prob

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
model_prob = model.predict_proba(X_test)
model_prob

In [ ]:
# lets see the +ve outcome:
model_prob = model_prob[:,1]
model_prob

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve


dummy_model_auc = roc_auc_score(y_test, dummy_model_prob)
model_auc = roc_auc_score(y_test, model_prob)
print(dummy_model_auc)
print(model_auc)



In [ ]:
dummy_fpr, dummy_tpr, _ = roc_curve(y_test, dummy_model_prob)
model_fpr, model_tpr, thresholds = roc_curve(y_test, model_prob)

In [ ]:
plt.plot(dummy_fpr, dummy_tpr, linestyle='--', label='Dummy')
plt.plot(model_fpr, model_tpr, marker='.', label='Logistic')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()

In [ ]:
thresholds

In [ ]:
# plotting roc curve for the model:
fig = plt.figure(figsize=(20,60))
plt.plot(dummy_fpr, dummy_tpr, linestyle='--', label='Dummy')
plt.plot(model_fpr, model_tpr, marker='.', label='Logistic')
for xyz in zip(model_fpr, model_tpr, thresholds):
    plt.annotate('%s' % np.round(xyz[2],2), xy = (xyz[0],xyz[1]))
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 10))  # smaller size is usually enough

# Plot dummy and model ROC curves
ax.plot(dummy_fpr, dummy_tpr, linestyle='--', label='Dummy')
ax.plot(model_fpr, model_tpr, marker='.', label='Logistic')

# Annotate threshold values
for fpr, tpr, threshold in zip(model_fpr, model_tpr, thresholds):
    ax.annotate(f'{threshold:.2f}', xy=(fpr, tpr), textcoords='offset points', xytext=(5,-5))

# Labels and legend
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()

# Show plot
plt.show()
